# Huấn luyện detector trên Kaggle

Chỉnh cấu hình ở mục 2–3 ngay trong notebook (không cần sửa `settings.py`), rồi chạy lần lượt các cell.

| | |
|---|---|
| **Model** | 1, 2 hoặc cả 3 kiến trúc (B-FATE, B-AVH, P2); mỗi kiến trúc chạy các seed đã chọn |
| **Báo cáo** | Riêng từng detector/seed trên **validation**, không tự xếp hạng. Test tuỳ chọn, tắt mặc định |
| **Kaggle** | Bật Internet + GPU |

> `compare.ipynb` chỉ là khung cho phần so sánh sau này. Worker/checkpoint thật vẫn cần kiểm chứng qua pilot Kaggle.

## 1. Cài đặt
Clone repository, cài `data_pipeline` và gói `[training]`.

In [ ]:
from pathlib import Path
import subprocess, sys, os
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
REPO_URL = "https://github.com/linhxm/vn-av-df-forensics.git"  # Sửa nếu đổi repository.
ROOT = Path("/kaggle/working/vn-av-df-forensics")
# Môi trường worker (~10+ GB) để ngoài /kaggle/working: không tính vào Output 20 GB, mất khi hết phiên.
WORKERS = Path("/kaggle/temp/vn-av-df-workers")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
assert (ROOT / "settings.py").is_file(), "Cần upload/push code mới trước khi dùng notebook"
sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT / "data_pipeline"), "-e", str(ROOT) + "[training]"], check=True)
import settings
settings.bootstrap()
from vn_av_df.actions import execute

## 2. Chọn part, kiến trúc và tên lần chạy
- `DATASET_PARTS` chọn dữ liệu, `ARCHITECTURES` chọn model: hai lựa chọn độc lập.
- `RUN_NAME` là folder output, ví dụ `train_part1_gru_run01`, `train_parts1_2_all_run01`.
- Đổi dữ liệu/model/hyperparameter thì dùng tên run mới. `RESUME` chỉ dùng khi tiếp tục đúng lần chạy bị ngắt.

In [ ]:
DATASET_ROOT = Path("/kaggle/input")
# Trỏ đến folder có manifest.jsonl + dataset_info.json; sửa đúng mount path Kaggle của bạn.
DATASET_PARTS = ["vn-av-df-data-part1/vn-av-df-data-part1"]
# Một part khác: ["vn-av-df-data-part2/vn-av-df-data-part2"]
# Hai part:
# DATASET_PARTS = ["vn-av-df-data-part1/vn-av-df-data-part1", "vn-av-df-data-part2/vn-av-df-data-part2"]
# DATASET_PARTS = "all"  # Mọi part đã attach, không tự download.
# Nếu tất cả nằm trong một Kaggle dataset:
# DATASET_ROOT = Path("/kaggle/input/YOUR_COLLECTION/vn-av-df-data")
# DATASET_PARTS = ["vn-av-df-data-part1", "vn-av-df-data-part2"]

ARCHITECTURES = "all"  # B-FATE (fate_gru), B-AVH (avh_tcn) và P2 (p2_syncartifact).
# ARCHITECTURES = ["fate_gru"]  # Chỉ B-FATE, không cần worker AV-HuBERT.
# ARCHITECTURES = ["p2_syncartifact"]  # Chỉ P2: AV-HuBERT + DINOv2, không tải FATE.
# Ablation P2 (dùng lại cache AV-HuBERT/DINOv2):
# ARCHITECTURES = ["avh_realrecon", "p2_sync_only", "p2_artifact_only", "p2_concat", "p2_sync_seen_fake"]
RUN_NAME = "train_part1_run01"
DEVICE = "cuda"
SEEDS = [42, 43, 44]  # Pilot: [42]. 3 kiến trúc x 3 seed = 9 lượt train detector.
RESUME = False
RUN_TEST = False  # True chỉ khi đã chốt cấu hình; không dùng test để chọn model.
CACHE_DIR = ROOT / "cache/vn-av-df-data"  # Cache có thể dùng lại nếu cùng media/encoder.
HELD_OUT_GENERATORS = ["musetalk_1_5"]  # Generator chỉ dùng test trong protocol chính.
# Pilot chỉ có Wav2Lip: đặt [] (đây là protocol khác; dùng RUN_NAME riêng).

## 3. Thông số train và trích feature
- `TRAINING` áp dụng cho detector được chọn; `RECONSTRUCTION` (stage A) và `SYNC` (stage S) chỉ dùng cho nhánh tương ứng.
- Linear không dùng hidden/dropout; training xử lý một clip mỗi bước tối ưu, không có batch size detector.
- Đổi thông số trích feature cần cache phù hợp và `RUN_NAME` mới.

In [ ]:
TRAINING = {
    "epochs": 20,             # Pilot: 2.
    "lr": 3e-4,              # Learning rate AdamW.
    "weight_decay": 1e-4,    # Regularization.
    "hidden": 128,           # Kích thước hidden/projection của temporal head.
    "dropout": 0.1,
    "patience": 5,           # Dừng sau 5 epoch validation loss không cải thiện; 0 = tắt.
    "top_fraction": 0.1,     # Score video = mean top 10% ô hợp lệ.
    "torch_threads": 2,      # Luồng CPU, không phải batch size.
    "balance_parents": True, # Mỗi epoch lấy một variant/parent để cân bằng nguồn.
    "artifact_aux_weight": 0.5,  # P2: trọng số head phụ artifact (nhãn AI theo ô).
}
RECONSTRUCTION = {
    "epochs": 20,            # Pilot: 2. Stage A của P1/P2: chỉ real train/validation.
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "patience": 5,
}
SYNC = {
    "epochs": 20,            # Pilot: 2. Stage S của P2: real, sham, real dịch lệch; không thấy fake.
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "patience": 5,
    "shift_frames": [3, 15],  # Dịch audio ±3–15 frame (0,12–0,6 s) để tạo mẫu lệch.
    "shift_probability": 0.5, # Tỷ lệ real được dịch lệch mỗi epoch.
}
FATE_FEATURES = {
    "precision": "float16",  # CPU: đổi thành float32.
    "window_s": 2.0, "step_s": 0.2, "temporal_bins": 8,
    "min_coverage": 0.75, "max_duration": 60, "max_side": 640,
    "batch_windows": 8,      # Số cửa sổ mỗi lượt model; chỉ đổi tốc độ/VRAM. Thiếu VRAM: giảm.
}
AVH_FEATURES = {
    "chunk_frames": 200, "overlap_frames": 50,
    "output_stride": 5,      # 25 Hz / 5 = output 5 Hz, mỗi ô 0.2 s.
    "max_duration": 60, "max_side": 640,
    "workers_per_device": 2,  # Worker mỗi GPU: dlib dò mặt chạy CPU một luồng (Kaggle 4 CPU, 2 GPU).
}
DINO_FEATURES = {
    "precision": "float16",  # CPU: đổi thành float32.
    "crop_size": 224,        # Crop miệng RGB cho DINOv2 ViT-S/14, bội số 14.
    "batch_size": 64,        # Giảm nếu thiếu VRAM.
}
BOOTSTRAP_REPLICATES = 1000  # CI khi chủ động chạy test; không ảnh hưởng train.

## 4. Áp cấu hình, tải encoder và worker
Kiểm tra dữ liệu (hash từng part), tải encoder của model đã chọn; worker AV-HuBERT chỉ cài khi kiến trúc cần.

In [ ]:
MAIN_METHODS = ["fate_gru", "avh_tcn", "p2_syncartifact"]
chosen = MAIN_METHODS.copy() if ARCHITECTURES == "all" else ARCHITECTURES
if not isinstance(chosen, list) or not chosen or len(set(chosen)) != len(chosen):
    raise ValueError("ARCHITECTURES phải là 'all' hoặc danh sách model không trùng")
settings.DATASET, settings.DATASET_PARTS = DATASET_ROOT, DATASET_PARTS
settings.ARCHITECTURES, settings.SEEDS = chosen, SEEDS
settings.RUN_NAME, settings.DEVICE = RUN_NAME, DEVICE
settings.DEMO_METHOD = chosen[0]  # Chỉ làm mặc định demo, không phải model tốt nhất.
cfg = settings.config()
if set(chosen) - set(cfg["methods"]):
    raise ValueError("Tên model không có trong settings.config()['methods']")
cfg["cache"] = str(CACHE_DIR)
cfg["held_out_generators"] = HELD_OUT_GENERATORS
cfg["training"].update(TRAINING)
cfg["reconstruction"].update(RECONSTRUCTION)
cfg["sync"].update(SYNC)
cfg["encoders"]["fate"].update(FATE_FEATURES)
cfg["encoders"]["avhubert"].update(AVH_FEATURES)
cfg["encoders"]["dinov2"].update(DINO_FEATURES)
cfg["bootstrap_replicates"] = BOOTSTRAP_REPLICATES
import torch
# Trích feature song song trên mọi GPU (mỗi GPU một encoder); cache giống nhau dù trích ở GPU nào.
cfg["prepare_devices"] = [f"cuda:{i}" for i in range(torch.cuda.device_count())] if DEVICE == "cuda" else None
from vn_av_df.dataset import training_dataset
rows, selection = training_dataset(cfg, verify_media=True)
print("Part và hash:", selection)
print("Model:", chosen, "| Seeds:", SEEDS, "| Output:", cfg["runs"])
execute("encoder_setup", cfg)  # Chỉ tải encoder của các model đã chọn.

In [ ]:
needs_avhubert = any(cfg["methods"][name]["encoder"] == "avhubert" for name in cfg["architectures"])
if needs_avhubert:
    try:
        subprocess.run([sys.executable, str(ROOT / "environments/bootstrap_worker.py"), "avhubert", "--root", str(ROOT), "--env-root", str(WORKERS)], check=True)
    except subprocess.CalledProcessError:
        # Lỗi thật (compiler, pip) nằm trong log cài đặt, không nằm trong CalledProcessError.
        log = WORKERS / ".kaggle-tools/avhubert/setup.log"
        if log.exists():
            lines = log.read_text(encoding="utf-8", errors="replace").splitlines()
            hits = [i for i, line in enumerate(lines) if "error" in line.lower()]
            start = max(0, hits[0] - 5) if hits else len(lines) - 80
            print("\n".join(lines[start:start + 60]))  # Đoạn quanh lỗi đầu tiên.
            print("...\n" + "\n".join(lines[-20:]))
        raise RuntimeError(f"Cài worker AV-HuBERT lỗi; xem đoạn log ở trên ({log})") from None
    worker_python = str(WORKERS / ".venv-avhubert/bin/python")
    settings.AVH_PYTHON = worker_python
    cfg["encoders"]["avhubert"]["python"] = worker_python
    print("Worker ready:", worker_python)
else:
    print("Các model đã chọn chỉ dùng FATE: bỏ qua worker AV-HuBERT.")

## 5. W&B (tuỳ chọn)
Mỗi bước một run, cùng group = `RUN_NAME`:
- `<RUN_NAME>-prepare`: dữ liệu đã chọn (số mẫu theo split × ô 2×2, theo speaker), log trích feature và thời gian từng encoder.
- `<RUN_NAME>-<detector>_seed<N>`: ghi **theo từng epoch** lúc train (stage A, stage S, detector: loss, AUC, F1, AUC theo ô 2×2, AUC từng nhánh P2, thời gian epoch) cùng log console và GPU/RAM.
- Mục 9 bổ sung metric validation/test, bảng theo nhóm, dự đoán từng video, ROC/PR và ảnh vào đúng run detector.
- Lần đầu: thêm Kaggle secret `WANDB_API_KEY` (*Add-ons → Secrets*, bật cho notebook).
- Tắt: đặt `USE_WANDB = False`. Không có secret thì tự tắt, pipeline vẫn chạy.

In [ ]:
import json
USE_WANDB = True
WANDB_PROJECT = "vn-av-df-forensics"
WANDB_ENTITY = None  # Team W&B; None = tài khoản của API key.
if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb
    try:
        from kaggle_secrets import UserSecretsClient
        # Key qua biến môi trường (không truyền vào code); W&B tự đọc.
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
        wandb.login()
    except Exception as exc:  # Không có secret: tắt W&B, pipeline vẫn chạy.
        USE_WANDB = False
        print("Tắt W&B, không đọc được secret WANDB_API_KEY:", exc)

def plain(value):
    """Cấu hình/metric về kiểu JSON đơn giản để W&B nhận."""
    return json.loads(json.dumps(value, default=str))

def stamp():
    """Ngày giờ Việt Nam (máy Kaggle chạy UTC) gắn vào tên run để phân biệt các lần chạy."""
    from datetime import datetime, timedelta, timezone
    return datetime.now(timezone(timedelta(hours=7))).strftime("%m%d-%H%M")

def table(rows):
    """Bảng W&B từ list dict; ô lồng (dict/list) đổi thành chuỗi JSON."""
    import pandas as pd
    frame = pd.DataFrame(rows)
    for column in frame.columns:
        if frame[column].map(lambda v: isinstance(v, (dict, list))).any():
            frame[column] = frame[column].map(lambda v: json.dumps(v, ensure_ascii=False))
    return wandb.Table(dataframe=frame)

def dataset_tables():
    """Dữ liệu đã chọn: số mẫu theo split × ô 2×2 và theo split × speaker (từ rows của mục 4)."""
    from collections import Counter
    from vn_av_df.dataset import condition
    cells = Counter((r["split"], condition(r)) for r in rows)
    speakers = {}
    for r in rows:
        item = speakers.setdefault((r["split"], r.get("speaker_id") or "-"), {"samples": 0, "minutes": 0.0, "sources": set()})
        item["samples"] += 1
        item["minutes"] += r["duration_s"] / 60
        item["sources"].add(r["source_id"])
    return {
        "data/by_condition": table([{"split": s, "condition": c, "samples": n} for (s, c), n in sorted(cells.items())]),
        "data/by_speaker": table([{"split": s, "speaker_id": p, "samples": v["samples"], "sources": len(v["sources"]),
                                    "minutes": round(v["minutes"], 2)} for (s, p), v in sorted(speakers.items())]),
    }

def start_prepare_run():
    """Run riêng cho trích feature: W&B ghi log console và GPU/RAM suốt bước prepare."""
    if not USE_WANDB:
        return None
    if wandb.run is not None:
        wandb.finish()
    run = wandb.init(
        project=WANDB_PROJECT, entity=WANDB_ENTITY, job_type="prepare", group=RUN_NAME,
        name=f"{RUN_NAME}-prepare-{stamp()}",
        config=plain({"dataset": selection, "architectures": chosen, "held_out_generators": HELD_OUT_GENERATORS,
                      "features": {"fate": FATE_FEATURES, "avhubert": AVH_FEATURES, "dinov2": DINO_FEATURES}}),
    )
    run.log(dataset_tables())
    return run

def finish_prepare_run(run):
    """Thời gian trích theo encoder (extraction.json của từng cache) rồi đóng run."""
    if run is None:
        return
    cache = Path(cfg["cache"])  # Prepare lỗi sớm có thể chưa tạo thư mục cache.
    for folder in sorted(cache.glob("*/extraction.json")) if cache.is_dir() else []:
        folder = folder.parent
        info = json.loads((folder / "extraction.json").read_text(encoding="utf-8"))
        run.summary.update({f"{folder.name}/samples": info["samples"],
                            f"{folder.name}/minutes": round(info["elapsed_s"] / 60, 2),
                            f"{folder.name}/seconds_per_sample": round(info["elapsed_s"] / max(1, info["samples"]), 2)})
    run.finish()

cfg.pop("wandb", None)
if USE_WANDB:  # Code train đọc cfg["wandb"] để ghi theo từng epoch (mỗi detector/seed một run).
    cfg["wandb"] = {"project": WANDB_PROJECT, "entity": WANDB_ENTITY, "group": RUN_NAME}

## 6. Trích feature và train
- Prepare dùng chung cache giữa các model cùng encoder; có thể trích feature test nhưng không fit thống kê hay train trên test.
- Phiên bị ngắt: giữ cache/run, chạy lại với `RESUME = True`, cùng dữ liệu/cấu hình/code.

In [ ]:
# W&B: run "<RUN_NAME>-prepare". Tắt W&B hoặc bỏ qua cell mục 5 thì không ghi, prepare vẫn chạy.
prepare_run = start_prepare_run() if globals().get("USE_WANDB") else None
try:
    execute("prepare", cfg)
finally:
    if prepare_run is not None:
        finish_prepare_run(prepare_run)

In [ ]:
execute("train", cfg, resume=RESUME)  # Chỉ train, không compare và không tự evaluate.

## 7. Báo cáo từng detector (validation)
- Loss/validation theo epoch; ROC, PR, confusion matrix và phân bố score của best checkpoint.
- Timeline dùng một real và một fake đầu tiên theo sample ID, không chọn theo độ đẹp của dự đoán.
- Validation đã dùng chọn checkpoint/ngưỡng, nên không coi là kết quả test độc lập.
- Chạy lại được riêng cell này từ thư mục artifact đã tải: chỉ cần JSON và thư viện vẽ, không cần GPU/video.

In [ ]:
from IPython.display import display, Markdown, Image
from vn_av_df.reporting import training_reports

def fmt(value):
    return f"{value:.4f}" if isinstance(value, float) else ("-" if value is None else str(value))

def show_detector_reports(split="validation"):
    # Mỗi vòng là một detector/seed, không tạo bảng so sánh nhiều detector.
    for item in training_reports(cfg["runs"], split=split):
        info = item["summary"]
        display(Markdown(f"### {info['architecture']} - seed {info['seed']} - {split}"))
        display(Markdown(f"Best epoch: **{info['best_epoch']}** / {info['epochs_completed']} epoch; "
                         f"video coverage: **{info['video_coverage']:.1%}**."))
        lines = ["| Chỉ số video | Giá trị |", "|---|---|"]
        for name, value in info["video_metrics"].items():
            if name != "confusion_matrix":
                lines.append(f"| {name} | {fmt(value)} |")
        display(Markdown("\n".join(lines)))
        # Ô 2×2: fake tính cùng real gốc; real/sham riêng chỉ có ý nghĩa ở false_alarm_rate.
        lines = ["| Ô | N | ROC-AUC | F1 | False alarm |", "|---|---|---|---|---|"]
        for cell, m in info["by_condition"].items():
            lines.append(f"| {cell} | {m['samples']} | {fmt(m['roc_auc'])} | {fmt(m['f1'])} | {fmt(m['false_alarm_rate'])} |")
        display(Markdown("\n".join(lines)))
        if info.get("branches"):
            # P2: AUC từng nhánh, real gốc (0) so với từng ô (1); dòng sham = độ nhạy lệch A/V.
            scores = info["branches"]["scores"]
            cells = sorted({c for v in scores.values() for c in v})
            lines = ["| Nhánh | " + " | ".join(cells) + " |", "|---" * (len(cells) + 1) + "|"]
            for name, values in scores.items():
                lines.append(f"| {name} | " + " | ".join(fmt(values.get(c, {}).get("roc_auc")) for c in cells) + " |")
            display(Markdown("\n".join(lines)))
        display(Markdown(f"Trainable parameters: **{info['resources']['trainable_parameters']:,}**; "
                         f"thời gian train trong phiên: **{info['resources']['elapsed_this_session_s']:.1f}s**."))
        if info.get("reconstruction"):
            display(Markdown(f"Stage A→V real-only - best validation loss: "
                             f"**{info['reconstruction']['best_validation_loss']:.4f}**; "
                             f"constant predictor: **{info['reconstruction']['constant_validation_loss']:.4f}**."))
        if info.get("sync"):
            display(Markdown(f"Stage S sync (không thấy fake) - best validation loss: "
                             f"**{info['sync']['best_validation_loss']:.4f}**; "
                             f"AUC real vs sham/dịch lệch: **{fmt(info['sync']['best_validation_auc'])}**."))
        for path in item["images"]:
            display(Image(filename=path))

show_detector_reports("validation")

## 8. Test (tuỳ chọn)
Mặc định bỏ qua. Khi `RUN_TEST = True`, đánh giá riêng từng detector với checkpoint/ngưỡng đã khóa trên validation.
Không xếp hạng hay so sánh cặp; chỉ lưu `test.json`, `predictions-test.json` và ảnh từng detector.

In [ ]:
if RUN_TEST:
    execute("evaluate", cfg)
    show_detector_reports("test")
else:
    print("Chưa chạy test. Báo cáo bên trên là validation.")

## 9. Bổ sung kết quả lên W&B
Mở lại run của từng detector/seed, thêm: kết quả stage A/S, metric best checkpoint (validation) và test (nếu có), temporal AP, ma trận nhầm lẫn,
bảng theo ô 2×2 / nhánh P2 / generator / variant / part, bảng dự đoán từng video, ROC/PR tương tác và ảnh báo cáo. Bỏ qua nếu `USE_WANDB = False`.

In [ ]:
# Bổ sung vào run của từng detector/seed (đã ghi theo epoch lúc train): kết quả stage A/S, metric
# best checkpoint và test (nếu RUN_TEST), bảng theo ô 2×2 / nhánh P2 / generator / variant / part,
# bảng dự đoán từng video, ROC/PR tương tác, ma trận nhầm lẫn và ảnh báo cáo.
if globals().get("USE_WANDB"):  # Bỏ qua cell mục 5 cũng coi như tắt W&B.
    runs = Path(cfg["runs"])

    def no_matrix(metrics):
        return {k: v for k, v in metrics.items() if k != "confusion_matrix"}

    for entry in json.loads((runs / "training.json").read_text(encoding="utf-8"))["runs"]:
        folder = runs / f"{entry['architecture']}_seed{entry['seed']}"

        def load(name):
            path = folder / name
            return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

        live = load("wandb.json")  # Run đã ghi theo epoch lúc train.
        if live:
            run = wandb.init(project=live["project"], entity=live["entity"] or None, id=live["id"], resume="must")
        else:  # Train khi tắt W&B: tạo run mới, ghi lịch sử từ file.
            run = wandb.init(
                project=WANDB_PROJECT, entity=WANDB_ENTITY, job_type="train", group=RUN_NAME,
                name=f"{RUN_NAME}-{folder.name}-{stamp()}",
                config=plain({"architecture": entry["architecture"], "seed": entry["seed"],
                              "dataset": selection, "training": TRAINING}),
            )
            # Mỗi stage có trục epoch riêng: stageA (tái dựng A→V), stageS (sync), detector.
            for stage, name in (("stageA", "reconstruction.json"), ("stageS", "sync.json"),
                                ("detector", "history.json")):
                data = load(name)
                history = data if isinstance(data, list) else (data or {}).get("history", [])
                run.define_metric(f"{stage}/epoch")
                run.define_metric(f"{stage}/*", step_metric=f"{stage}/epoch")
                for row in history:
                    run.log({f"{stage}/{k}": v for k, v in row.items() if v is not None})
        best = load("best.json") or {}
        run.summary.update(plain({"best_epoch": best.get("epoch"), "thresholds": best.get("thresholds")}))
        # Stage A: R tốt hơn dự đoán hằng số bao nhiêu; stage S: AUC real vs sham/dịch lệch.
        stage_a, stage_s = load("reconstruction.json"), load("sync.json")
        if stage_a:
            run.summary.update({"stageA/best_validation_loss": stage_a["best_validation_loss"],
                                "stageA/constant_validation_loss": stage_a["constant_validation_loss"],
                                "stageA/real_train": len(stage_a["train_ids"]),
                                "stageA/real_validation": len(stage_a["validation_ids"])})
        if stage_s:
            run.summary.update(plain({"stageS/best_validation_loss": stage_s["best_validation_loss"],
                                      "stageS/best_validation_auc": stage_s["best_validation_auc"],
                                      "stageS/sham_train": stage_s["sham_train"],
                                      "stageS/sham_validation": stage_s["sham_validation"]}))
        thresholds = best.get("thresholds") or {}
        for split in ("validation", "test"):  # test chỉ có khi RUN_TEST=True.
            metrics = load(f"{split}.json")
            if not metrics:
                continue
            video = metrics["video"]
            run.summary.update(plain({f"{split}/{k}": v for k, v in no_matrix(video).items()}))
            run.summary.update(plain({f"{split}/{k}": metrics.get(k) for k in (
                "video_coverage", "window_coverage", "temporal_labeled_videos",
                "real_video_interval_false_alarm_rate", "false_intervals_per_real_hour",
                "head_inference_seconds")}))
            for iou_threshold, values in (metrics.get("temporal") or {}).items():
                run.summary.update(plain({f"{split}/temporal_iou{iou_threshold}/{k}": v for k, v in values.items()}))
            (tn, fp), (fn, tp) = video["confusion_matrix"]
            run.summary.update({f"{split}/tn": tn, f"{split}/fp": fp, f"{split}/fn": fn, f"{split}/tp": tp})
            if metrics.get("uncertainty"):
                run.summary.update(plain({f"{split}/uncertainty": metrics["uncertainty"]}))
            # Bảng theo nhóm: ô 2×2 (mọi split), generator/variant/part (chỉ test).
            for key in ("by_condition", "by_generator", "by_variant", "by_part"):
                groups = metrics.get(key) or {}
                if groups:
                    run.log({f"{split}/{key}": table([{"group": g, **no_matrix(m["video"])} for g, m in groups.items()])})
            scores = (metrics.get("branches") or {}).get("scores") or {}
            branch_rows = [{"branch": b, "cell": c, **v} for b, values in scores.items() for c, v in values.items()]
            if branch_rows:
                run.log({f"{split}/branches": table(branch_rows)})
            # Dự đoán từng video (không gồm timeline) + ROC/PR tương tác để xem lỗi theo mẫu/ô.
            predictions = load(f"predictions-{split}.json") or []
            measured = [p for p in predictions if p["video_score"] is not None]
            if measured:
                run.log({f"{split}/predictions": table([
                    {k: p.get(k) for k in ("sample_id", "label", "condition", "dataset_part", "source_id",
                                           "duration_s", "video_score")}
                    | {"predicted": int(p["video_score"] >= thresholds.get("video", 0.5))}
                    for p in measured])})
                labels = [p["label"] for p in measured]
                if len(set(labels)) == 2:
                    probas = [[1 - p["video_score"], p["video_score"]] for p in measured]
                    run.log({f"{split}/roc": wandb.plot.roc_curve(labels, probas, labels=["real", "fake"], classes_to_plot=[1]),
                             f"{split}/pr": wandb.plot.pr_curve(labels, probas, labels=["real", "fake"], classes_to_plot=[1])})
        resources = load("resources.json") or {}
        run.summary.update(plain({f"resources/{k}": v for k, v in resources.items()
                                  if isinstance(v, (int, float))}))
        images = sorted((folder / "report").glob("*.png"))  # Có sau khi chạy cell báo cáo.
        if images:
            run.log({"report": [wandb.Image(str(p), caption=p.stem) for p in images]})
        run.finish()

## 10. Dọn Output
Output của notebook (/kaggle/working, tối đa 20 GB) giữ nguyên thư mục run, không nén ZIP (ZIP chỉ là bản sao, tốn gấp đôi chỗ):
- `runs/<RUN_NAME>/`: checkpoint, cấu hình, part/hash, dự đoán, hình báo cáo và thông tin worker AV-HuBERT (`environment/avhubert/`).
- `cache/`: feature đã trích (ước tính ~1–2 MB/video), giữ để train lại/ablation không phải trích lại; `KEEP_CACHE = False` để xoá.
- Xoá `external/`, `weights/` (source/weights encoder tải lại được).

Sau đó *Save Version*; `compare.ipynb` sau này attach Output này (thư mục `runs/<RUN_NAME>`).

In [ ]:
import shutil
KEEP_CACHE = True  # False: xoá cache feature khỏi Output (phiên sau phải trích lại).
runs = Path(cfg["runs"])
# Thông tin worker (receipt, pip freeze, log cài) đi cùng run; WORKERS ở /kaggle/temp sẽ mất khi hết phiên.
if needs_avhubert:
    target = runs / "environment/avhubert"
    target.mkdir(parents=True, exist_ok=True)
    for p in (WORKERS / ".kaggle-tools/avhubert").glob("*"):
        if p.is_file():
            shutil.copy2(p, target / p.name)
for name in ("external", "weights") + (() if KEEP_CACHE else ("cache",)):
    path = ROOT / name
    if path.exists():
        shutil.rmtree(path, ignore_errors=True)
        print("Đã xoá", path)

def size_gb(path):
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / 1e9 if path.exists() else 0.0

print(f"Run: {runs} ({size_gb(runs):.2f} GB)")
print(f"Cache: {cfg['cache']} ({size_gb(Path(cfg['cache'])):.2f} GB)")
print(f"Tổng /kaggle/working: {size_gb(Path('/kaggle/working')):.2f} GB (giới hạn Output 20 GB)")